# Fase 4 — Validasi Lintas-Dataset di SWRD RESMI (B2 vs P1)

Menjawab pertanyaan reviewer paling mendasar: **apakah temuan di steel-pipe (DSF-YOLO) berlaku juga di dataset lain?**

- Dataset: **SWRD resmi, tile v1** (Kaggle `benyxu/swrd-dataset`; dibangun dari `SWXD_Data` dgn `build_swrd_tiles.py` **v4.1**)
  - 3.400 tile PNG grayscale 8-bit ≤640×640 (train 2.700 = 1.350 positif + 1.350 negatif; val 700 = 350 + 350), **6 kelas**
  - split per batch tanggal 90/10 (`swrd_split_v1.csv`, seed 7) — tidak ada film/batch yg muncul di train & val
  - label = kotak bagian poligon cacat resmi di dalam tile; audit `audit_labels.py` LULUS, `check_tiles.py` LULUS
  - protokol lengkap: `claude/swrd-tiles-v1-protokol.md` (Project)
- ⚠️ **BUKAN dataset "swrd" lama** (= Roboflow weld_detection / RF-Weld, 5 kelas). Supaya hasil lama di Drive TIDAK
  tertimpa / ter-resume, semua run & CSV di notebook ini memakai penanda **`swrdv1`**.
- `deform_ss2d.py` = versi **GRID-FIX** (sha256 `675445871a2e4d55…d9aa`), dikunci lewat tripwire sha256 di Sel 2 —
  sama dgn P1-fix steel-pipe (5 seed). B2 tidak memakai modul ini sama sekali.
- Model: **B2 dan P1 saja** (10 run = 2 model × 5 seed). Resep **identik** steel-pipe (`SGD`, `lr0=0.002`, `cos_lr=False`,
  150 epoch kontinu, `patience=0`, `cache=True`, `amp=True`, batch 8, imgsz 640).
- Lingkungan dipin: **torch 2.11.0+cu128**, GPU **A100** (sama dgn run P1-fix/Fase 10–12) — lihat Sel 0.
- Endpoint utama: mAP50-95 enam kelas. **LoP (38 film val), LoF (29), undercut (9) = deskriptif** (<40 film val).

Catatan LR: jumlah citra train (2.700) ≈ steel-pipe (~2.700), jadi langkah gradien per epoch setara — *mismatch* skala
yg dulu dikhawatirkan utk RF-Weld (~2× lebih besar) tidak berlaku di sini. Tetap **pantau `cls_loss` ~20 epoch pertama run
pertama** — kalau naik terus/NaN, hentikan & laporkan.

## Progres

| # | Model | Seed | Status |
|---|---|---|---|
| 1 | B2 | 1 | ⬜ |
| 2 | B2 | 2 | ⬜ |
| 3 | B2 | 3 | ⬜ |
| 4 | B2 | 4 | ⬜ |
| 5 | B2 | 5 | ⬜ |
| 6 | P1 | 1 | ⬜ |
| 7 | P1 | 2 | ⬜ |
| 8 | P1 | 3 | ⬜ |
| 9 | P1 | 4 | ⬜ |
| 10 | P1 | 5 | ⬜ |


## 0. Pin Lingkungan — torch 2.11.0+cu128 (WAJIB paling awal, sebelum `import torch`)

Colab kini memasang torch **2.11.0+cu130** (CUDA 13). Kernel `selective_scan` gagal di-compile di CUDA 13, dan
run P1-fix/Fase 10–12 sebelumnya memakai **2.11.0+cu128**. Sel ini memasang build cu128 (versi torch SAMA, beda
build CUDA) lalu me-RESTART runtime otomatis. **Setelah restart: Runtime → Run all SEKALI LAGI** — kali kedua sel ini
hanya mencetak "OK" dan notebook lanjut, memakai wheel `selective_scan` cu128 yang sudah ada di cache Drive.


In [1]:
import importlib.metadata as _md, subprocess, sys, os
TORCH_PIN = "2.11.0+cu128"          # = lingkungan run P1-fix / Fase 10-12 (tracker: torch 2.11.0+cu128, py3.13)
_tv = _md.version("torch")
print("torch terpasang:", _tv)
if _tv != TORCH_PIN:
    assert "torch" not in sys.modules, "torch sudah ter-import di sesi ini -- Runtime > Restart session, lalu Run all"
    _vis = _md.version("torchvision").split("+")[0]      # versi torchvision pasangan torch ini, build cu128
    _pkgs = [f"torch=={TORCH_PIN}", f"torchvision=={_vis}+cu128"]
    print("Memasang", _pkgs, "dari indeks PyTorch cu128 (sekali per sesi, beberapa menit)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs,
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    print("Terpasang:", subprocess.run([sys.executable, "-c", "import torch,torchvision;print(torch.__version__, torch.version.cuda, torchvision.__version__)"],
                                       capture_output=True, text=True).stdout.strip())
    print("\n>>> Runtime akan RESTART sekarang. Setelah tersambung lagi: Runtime > Run all SEKALI LAGI. <<<")
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
else:
    print(f"OK: torch = {TORCH_PIN} -- lanjut.")


torch terpasang: 2.11.0+cu128
OK: torch = 2.11.0+cu128 -- lanjut.


## 1. Mount Drive + Kredensial Kaggle (Cache)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, stat, shutil

DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs(DRIVE_ROOT, exist_ok=True)
KAGGLE_JSON_CACHE = f"{DRIVE_ROOT}/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)
assert os.path.exists(KAGGLE_JSON_CACHE), (
    f"kaggle.json tidak ada di {KAGGLE_JSON_CACHE} -- jalankan fase-setup-colab-mamba.ipynb dulu."
)
shutil.copy(KAGGLE_JSON_CACHE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
print("kaggle.json dimuat dari cache Drive.")

Mounted at /content/drive
kaggle.json dimuat dari cache Drive.


## 2. Tarik Dataset SWRD v1 dari Kaggle + Tripwire Isi Dataset + Deteksi GPU


In [ ]:
import subprocess, sys, glob, torch

# Kontrol numerik lintas-GPU: cudnn.allow_tf32 default TRUE di torch, tapi TF32 cuma ada
# di hardware Ampere+ (A100, L4) -- TIDAK ADA di Turing (T4). Artinya B2/P1 seed0 (Kaggle T4)
# dan seed1+ (Colab, GPU bisa A100/L4) TIDAK menjalankan presisi numerik yang identik kalau
# dibiarkan default -- variabel yang belum terkontrol. Dimatikan di sini demi konsistensi
# lintas-platform, ditemukan setelah B2 seed1 (A100) turun di 7/8 kelas vs seed0 (T4).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print(f"TF32 dimatikan (matmul={torch.backends.cuda.matmul.allow_tf32}, cudnn={torch.backends.cudnn.allow_tf32})")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/deform_src", exist_ok=True)

# SWRD RESMI tile v1 (BUKAN 'swrd-yolo8format' = RF-Weld lama). Diunduh ke folder sendiri.
SWRD_DIR = "/content/data/swrd_v1"
if os.path.exists(SWRD_DIR):
    shutil.rmtree(SWRD_DIR)   # selalu unduh bersih -- jangan pakai sisa unduhan setengah jadi
os.makedirs(SWRD_DIR)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/swrd-dataset",
                 "-p", SWRD_DIR, "--unzip"], check=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/deform-ss2d-py",
                 "-p", "/content/deform_src", "--unzip"], check=True)

DEFORM_SRC = glob.glob("/content/deform_src/**/deform_ss2d.py", recursive=True)[0]

# ---------- TRIPWIRE VERSI deform_ss2d.py (sha256) ----------
# Ada TIGA versi di riwayat proyek; tripwire lama (x_soft/ssm_ratio/nama kelas) LOLOS untuk ketiganya:
#   cfed78de317a7781... = PRA-perbaikan (grid & path_ref tergeser +0.5 px) -> hasil primer steel-pipe lama
#   675445871a2e4d55... = GRID-FIX (dipakai P1-fix steel-pipe 5 seed, Fase 10/11)   <- DIPAKAI DI SINI
#   51520b33f9f77f28... = v2 jalur-terbatas (Fase 12, gagal) -- JANGAN
import hashlib
DEFORM_SHA256_EXPECTED = "675445871a2e4d55400175ef5f6fda83c90ece651bf818e3f54481bc14a6d9aa"
_raw = open(DEFORM_SRC, "rb").read()
DEFORM_SHA256 = hashlib.sha256(_raw).hexdigest()
_norm = hashlib.sha256(_raw.replace(b"\r\n", b"\n")).hexdigest()   # toleransi akhir-baris Windows saja
_txt = _raw.decode("utf-8", "replace")
print(f"deform_ss2d.py sha256 = {DEFORM_SHA256}")
if "linspace(0.5, H - 0.5" in _txt or "linspace(0.5, N - 0.5" in _txt:
    raise RuntimeError("deform_ss2d.py = versi PRA-PERBAIKAN (grid +0.5 px). Unggah versi grid-fix ke Kaggle 'deform-ss2d-py'.")
if "bounded_path_positions" in _txt:
    raise RuntimeError("deform_ss2d.py = versi v2 jalur-terbatas (Fase 12). Bukan yang dipakai di sini.")
assert DEFORM_SHA256 == DEFORM_SHA256_EXPECTED or _norm == DEFORM_SHA256_EXPECTED, (
    f"deform_ss2d.py BUKAN versi grid-fix yang tercatat (sha256 {DEFORM_SHA256[:16]}..., harus 675445871a2e4d55...). STOP.")
print("Tripwire versi: deform_ss2d.py = GRID-FIX (675445871a2e4d55...) -- OK")

# data.yaml dicari HANYA di folder SWRD v1 (yang paling dangkal = root dataset).
_yamls = sorted(glob.glob(f"{SWRD_DIR}/**/data.yaml", recursive=True), key=lambda p: p.count("/"))
assert _yamls, f"Tidak ada data.yaml di {SWRD_DIR} -- cek hasil unduhan Kaggle"
SWRD_YAML_SRC = _yamls[0]
SWRD_BASE = os.path.dirname(SWRD_YAML_SRC)
print(f"DEFORM_SRC    = {DEFORM_SRC}")
print(f"SWRD_YAML_SRC = {SWRD_YAML_SRC}")
print(f"SWRD_BASE     = {SWRD_BASE}")
print("\nIsi data.yaml bawaan:")
print(open(SWRD_YAML_SRC).read())
print("Isi folder dataset:")
for _p in sorted(glob.glob(f"{SWRD_BASE}/*"))[:15]:
    print("  ", _p)

# ---------- TRIPWIRE: pastikan yang terunduh PERSIS dataset v4.1 yang sudah diaudit ----------
# Angka acuan = hasil audit_labels.py (LULUS) & check_tiles.py (LULUS) di laptop, 14 Okt 2026.
# Kalau versi lain (mis. v3 atau RF-Weld) yang ter-upload, sel ini GAGAL KERAS sebelum build/training.
import yaml as _yaml
from collections import Counter as _Counter
SWRD_NAMES_EXPECTED = ["porosity", "inclusion", "crack", "undercut", "lack_of_fusion", "lack_of_penetration"]
_n = _yaml.safe_load(open(SWRD_YAML_SRC))["names"]
_n = [_n[k] for k in sorted(_n)] if isinstance(_n, dict) else list(_n)
assert _n == SWRD_NAMES_EXPECTED, f"Urutan/nama kelas beda: {_n}"
EXPECTED = {  # split: (jumlah tile, tile positif, objek per kelas [por, incl, crack, undercut, LoF, LoP])
    "train": (2700, 1350, [2204, 254, 244, 124, 130, 198]),
    "val":   (700,  350,  [619,  78,  85,  14,  47,  41]),
}
for _sp, (_nt, _npos, _nobj) in EXPECTED.items():
    _imgs = sorted(glob.glob(f"{SWRD_BASE}/images/{_sp}/*.png"))
    _labs = sorted(glob.glob(f"{SWRD_BASE}/labels/{_sp}/*.txt"))
    assert len(_imgs) == _nt and len(_labs) == _nt, f"{_sp}: {len(_imgs)} PNG / {len(_labs)} label, harus {_nt}"
    assert {os.path.splitext(os.path.basename(p))[0] for p in _imgs} == {os.path.splitext(os.path.basename(p))[0] for p in _labs}, f"{_sp}: nama PNG != label"
    _cnt, _pos = _Counter(), 0
    for _p in _labs:
        _rows = [l.split() for l in open(_p).read().splitlines() if l.strip()]
        _pos += bool(_rows)
        for _r in _rows:
            assert len(_r) == 5 and 0 <= int(_r[0]) < 6, f"label rusak: {_p}"
            _cnt[int(_r[0])] += 1
    _got = [_cnt[i] for i in range(6)]
    assert _pos == _npos and _got == _nobj, f"{_sp}: positif {_pos} (harus {_npos}), objek/kelas {_got} (harus {_nobj})"
    assert not glob.glob(f"{SWRD_BASE}/labels/{_sp}*.cache"), "ada .cache lama di dataset -- unggah ulang tanpa .cache"
    print(f"Tripwire {_sp}: {_nt} tile, {_npos} positif, objek/kelas {dict(zip(SWRD_NAMES_EXPECTED, _got))} -- COCOK v4.1")

print(f"\nGPU: {torch.cuda.get_device_name(0)}  (compute cap {torch.cuda.get_device_capability(0)})")
print("(GPU cepat -> mungkin selesai 1 sesi. GPU lambat/T4 -> auto-resume kalau kepotong, lihat Sel 7)")

TF32 dimatikan (matmul=False, cudnn=False)
deform_ss2d.py sha256 = 675445871a2e4d55400175ef5f6fda83c90ece651bf818e3f54481bc14a6d9aa
Tripwire versi: deform_ss2d.py = GRID-FIX (675445871a2e4d55...) -- OK
DEFORM_SRC    = /content/deform_src/deform_ss2d.py
SWRD_YAML_SRC = /content/data/swrd_v1/data.yaml
SWRD_BASE     = /content/data/swrd_v1

Isi data.yaml bawaan:
# path: diisi saat dipakai (mis. /kaggle/input/swrd-tiles-v1)
train: images/train
val: images/val
names:
  0: porosity
  1: inclusion
  2: crack
  3: undercut
  4: lack_of_fusion
  5: lack_of_penetration

Isi folder dataset:
   /content/data/swrd_v1/data.yaml
   /content/data/swrd_v1/images
   /content/data/swrd_v1/labels
   /content/data/swrd_v1/manifest.csv
   /content/data/swrd_v1/overlay_sheet.png
   /content/data/swrd_v1/summary.txt
Tripwire train: 2700 tile, 1350 positif, objek/kelas {'porosity': 2204, 'inclusion': 254, 'crack': 244, 'undercut': 124, 'lack_of_fusion': 130, 'lack_of_penetration': 198} -- COCOK v4.1
Tripwire v

## 3. 🔧 KONFIGURASI — Edit Ini Setiap Kali Ganti Kombinasi

In [ ]:
# ============================================================
#  UBAH BARIS INI SEBELUM "Run all"
# ============================================================
MODEL = "P1"   # HANYA "B2" atau "P1" di fase ini (validasi lintas-dataset, 10 run total).
               # Varian Ablasi B (BackboneOnly dst) TIDAK diikutkan -- cukup B2-vs-P1 utk
               # menjawab "apakah temuan DSF-YOLO berlaku di dataset lain?".
SEED  = 5      # 1, 2, 3, 4, atau 5 -- sama seperti protokol DSF-YOLO.
LR0 = 0.002   # WAJIB diturunkan dari default 0.01 -- lr0=0.01 terbukti tidak stabil utk P1+STE.
              # Bukti: linear rusak @epoch82 (LR saat itu ~0.0046), kosinus rusak LEBIH CEPAT
              # @ epoch7 (LR saat itu ~0.00995, msh dekat lr0 penuh -- kosinus menahan LR tinggi
              # lebih lama di awal, bukan menurunkannya lebih cepat spt diduga sebelumnya).
              # Ambang bahaya empiris ~0.0046-0.0054. lr0=0.002 dimulai DI BAWAH ambang itu
              # sejak epoch 1, bukan menurun ke arahnya -- margin aman, bukan mepet.
              # Dipakai jg utk BackboneOnly/NeckOnly (sama-sama punya blok DeformSS2D dgn STE,
              # belum ada bukti ambang beda -- pakai yg sudah terbukti aman utk P1 dulu).
USE_COS_LR = False   # KEMBALI ke linear -- kosinus TERBUKTI LEBIH BURUK (rusak epoch 7 vs 70+),
                     # krn kurvanya datar di awal (menahan LR dekat lr0 lebih lama), padahal yang
                     # dibutuhkan adalah LR rendah SEJAK AWAL, bukan cuma di akhir. Linear + LR0
                     # rendah menyerang akar masalah (nilai LR), bukan cuma bentuk kurvanya.
DIAG_LR = False   # True HANYA utk SATU run diagnostik: B2 seed 1, lr0=0.01 (default Ultralytics).
                  # BUKAN bagian perbandingan B2 vs P1 -- menguji apakah underfitting B2 di SWRD v1
                  # (mAP50:95 train 0.107 / val 0.062, Sel 9b) dibatasi oleh LR terkunci 0.002.
                  # Run, folder & baris CSV-nya dipisah total dari run utama (lihat CSV_TAG di bawah).
# ============================================================

TARGET_EPOCHS = 150
if DIAG_LR:
    assert MODEL == "B2" and SEED == 1, "Run diagnostik LR hanya B2 seed 1"
    LR0 = 0.01

# Peta konfigurasi per varian -- YAML, target params (hasil verifikasi capacity-matching
# offline, lihat tracker), jumlah DeformSS2D yg diharapkan (utk pre-flight Sel 5).
MODEL_CONFIG = {
    "B2":           {"yaml": "Mamba-YOLO-T.yaml",                    "params": 5.987, "ndef": 0},
    "P1":           {"yaml": "DeMamba-YOLO-T.yaml",                  "params": 5.931, "ndef": 10},
    # Varian Ablasi B sengaja TIDAK dipakai di fase ini (lihat catatan MODEL di atas).
    "BackboneOnly": {"yaml": "DeMamba-YOLO-T-backbone-only.yaml",    "params": 5.952, "ndef": 6},
    "NeckOnly":     {"yaml": "DeMamba-YOLO-T-neck-only.yaml",        "params": 5.966, "ndef": 4},
    # Ablasi B-lanjut: stage-wise (lihat literatur-deformable-lintas-paradigma.md SS6.3).
    # Neck dibiarkan B2 di semua varian ini -- pertanyaan yg diuji adalah "stage backbone
    # mana", bukan neck lagi (isolasi bersih dari pertanyaan Ablasi B sebelumnya).
    "BB-Deep":      {"yaml": "DeMamba-YOLO-T-bb-deep.yaml",          "params": 5.991, "ndef": 1},
    "BB-Late":      {"yaml": "DeMamba-YOLO-T-bb-late.yaml",          "params": 5.964, "ndef": 4},
    "BB-Early":     {"yaml": "DeMamba-YOLO-T-bb-early.yaml",         "params": 5.975, "ndef": 2},
    # BERSYARAT (backlog) -- lihat tracker Section 2d. Sumbu KETIGA: kerapatan intra-stage.
    # Posisi 5 = pola D-S-D (2/3 blok deform, 1 tetap biasa), posisi 7 = deform penuh.
    # Neck TIDAK disentuh (sama seperti semua varian BB-* lainnya).
    "BB-Late-Alt":  {"yaml": "DeMamba-YOLO-T-bb-late-alt.yaml",      "params": 5.973, "ndef": 3},
}
assert MODEL in ("B2", "P1"), f"Fase ini HANYA utk B2/P1, dapat: {MODEL}"
assert SEED in (1, 2, 3, 4, 5), f"SEED harus 1-5, dapat: {SEED}"

DATASET = "swrdv1" # penanda dataset -- MASUK ke RUN_NAME & nama file CSV. WAJIB "swrdv1", BUKAN "swrd":
                   # "swrd" sudah dipakai run RF-Weld (Roboflow) lama di Drive -> RUN_NAME sama akan
                   # membuat Sel 6 membaca checkpoint LAMA (resume/skip) & CSV lama tertimpa.
assert DATASET == "swrdv1"
RUN_NAME = f"{DATASET}_{MODEL.lower()}_seed{SEED}_150ep"
if USE_COS_LR:
    RUN_NAME += "_coslr"
if LR0 != 0.01:
    RUN_NAME += f"_lr{LR0}"
if DIAG_LR:
    RUN_NAME = f"{DATASET}_diag_{MODEL.lower()}_seed{SEED}_150ep_lr{LR0}"
# CSV_TAG = isi kolom "dataset" di CSV Sel 9/9b. Run utama: "swrdv1" (tidak berubah). Run diagnostik:
# tag lain + file CSV lain, supaya kunci dedup (dataset, model, seed, class) TIDAK menimpa B2 seed 1 utama.
CSV_TAG = f"{DATASET}_diag_lr{LR0}" if DIAG_LR else DATASET
# nama run beda kalau konfigurasi beda -- supaya TIDAK ke-resume checkpoint dgn train_args lama
# yang tidak sesuai (mis. lr0/cos_lr beda), harus mulai training dari nol tiap konfigurasi baru
CONFIG_YAML = MODEL_CONFIG[MODEL]["yaml"]
EXPECTED_PARAMS = MODEL_CONFIG[MODEL]["params"]
EXPECTED_NDEF = MODEL_CONFIG[MODEL]["ndef"]
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"DATASET={DATASET}  MODEL={MODEL}  SEED={SEED}  RUN_NAME={RUN_NAME}")
print(f"target={TARGET_EPOCHS} epoch  lr0={LR0}  cos_lr={USE_COS_LR}")
if DIAG_LR:
    print(">>> MODE DIAGNOSTIK LR -- BUKAN run perbandingan. CSV_TAG =", CSV_TAG, "<<<")


DATASET=swrdv1  MODEL=P1  SEED=5  RUN_NAME=swrdv1_p1_seed5_150ep_lr0.002
target=150 epoch  lr0=0.002  cos_lr=False


## 4. Clone Mamba-YOLO + Semua Patch + Build Kernel

Identik Fase 2A-Colab/2B-Colab -- 8 patch registrasi + 2 patch `weights_only` (`tasks.py`) +
1 patch `weights_only` (`torch_utils.py`), kernel vendor-only.

In [ ]:
import re, py_compile

REPO = "/content/Mamba-YOLO"
# WAJIB sebelum rmtree: pastikan cwd proses TIDAK sedang di dalam REPO (bisa terjadi kalau
# notebook di-run ulang di kernel yang sama setelah sel pre-flight sempat os.chdir(REPO) --
# menghapus folder yang sedang "ditempati" proses bikin SEMUA subprocess berikutnya gagal
# dgn "fatal: Unable to read current working directory").
os.chdir("/content")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HZAI-ZJNU/Mamba-YOLO.git", REPO], check=True)

DEFORM = f"{REPO}/ultralytics/nn/modules/deform_ss2d.py"
shutil.copy(DEFORM_SRC, DEFORM)

src = open(DEFORM).read()
checks = {
    "STE (gradien Delta-p)": "x_soft" in src,
    "ssm_ratio=1.4 (capacity match P1)": "ssm_ratio=1.4" in src,
    "subclass block terdaftar": "class DeformVSSBlock" in src and "class DeformXSSBlock" in src,
}
if not all(checks.values()):
    raise RuntimeError("deform_ss2d.py BUKAN versi final -- cek dataset 'deform-ss2d-py' di Kaggle.")
print("Tripwire deform_ss2d.py OK")

def patch(path, subs):
    p = f"{REPO}/{path}"
    src = open(p).read()
    for old, new, tag in subs:
        if new in src:
            print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new)
        print(f"  OK {tag}")
    open(p, "w").write(src)

print("\n### Patch")
patch("ultralytics/nn/modules/mamba_yolo.py", [
    ("class XSSBlock(nn.Module):\n    def __init__(",
     "class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.1"),
    ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,",
     "nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,", "1.2"),
    ("class VSSBlock(nn.Module):\n    def __init__(",
     "class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.3"),
    ("            self.op = SS2D(",
     "            self.op = self.SSM_CLASS(", "1.4"),
])
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("from .mamba_yolo import SS2D",
     "from .mamba_yolo import SS2D, VSSBlock, XSSBlock", "2.1"),
])
patch("ultralytics/nn/tasks.py", [
    ("    XSSBlock\n)\n",
     "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import DeformVSSBlock, DeformXSSBlock\n", "3.1"),
    ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
     "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock\n        }:", "3.2"),
    ("RepC3, XSSBlock}:",
     "RepC3, XSSBlock, DeformXSSBlock}:", "3.3"),
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("ultralytics/utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
y = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", y)
y = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml", "w").write(y)

# Ablasi B: 2 varian campuran -- backbone-saja-deformable, neck-saja-deformable.
# Split di penanda "# Mamba-YOLO PAFPN" (awal section head/neck), regex-swap CUMA
# di satu bagian. Params sudah diverifikasi offline: BackboneOnly=5.952M(-0.58%),
# NeckOnly=5.966M(-0.36%) vs B2 -- keduanya dlm toleransi +-5% TANPA ssm_ratio baru.
base_yaml = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
split_marker = "# Mamba-YOLO PAFPN"
idx = base_yaml.find(split_marker)
assert idx != -1, "penanda split YAML tidak ketemu -- struktur Mamba-YOLO-T.yaml berubah?"
backbone_part, head_part = base_yaml[:idx], base_yaml[idx:]
assert "VSSBlock" in backbone_part and "XSSBlock" not in backbone_part
assert "XSSBlock" in head_part and "VSSBlock" not in head_part

backbone_deform = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", backbone_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-backbone-only.yaml", "w").write(
    backbone_deform + head_part)

head_deform = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", head_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-neck-only.yaml", "w").write(
    backbone_part + head_deform)
print("  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate")

# Ablasi B-lanjut: 3 varian stage-wise di backbone (literatur-deformable-lintas-paradigma.md
# SS6.3). Neck TIDAK disentuh (tetap XSSBlock biasa di semua 3 varian ini) -- isolasi bersih
# dari pertanyaan Ablasi B sebelumnya. Tiap baris backbone punya komentar posisi unik
# ("# 1","# 3","# 5","# 7") dipakai utk menarget baris tepat, assert-guarded (fail loud
# kalau match != 1x, mis. krn upstream ubah format YAML).
def _swap_positions(text, positions):
    lines = text.split("\n")
    hit = {p: 0 for p in positions}
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        for p in positions:
            if stripped.endswith(f"# {p}") and "VSSBlock" in line and "Deform" not in line:
                lines[i] = line.replace("VSSBlock", "DeformVSSBlock")
                hit[p] += 1
    for p in positions:
        assert hit[p] == 1, f"posisi {p}: ditemukan {hit[p]}x (harus tepat 1x) -- format YAML berubah?"
    return "\n".join(lines)

stagewise_variants = {"bb-deep": [7], "bb-late": [5, 7], "bb-early": [1, 3]}
for vname, positions in stagewise_variants.items():
    result = _swap_positions(base_yaml, positions)
    open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-{vname}.yaml", "w").write(result)
print(f"  OK {len(stagewise_variants)} YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate")

# BB-Late-Alt (BERSYARAT, backlog Section 2d) -- sumbu kerapatan intra-stage. Posisi 5
# dipecah jadi 3 baris terpisah (D-S-D) krn parse_model bikin n INSTANCE TERPISAH dari
# kelas yg sama per baris (bukan loop internal) -- lihat tasks.py baris ~946. Memecah 1
# baris jadi 3 menggeser SEMUA indeks "from" eksplisit di bawahnya +2 -- WAJIB diperbaiki,
# assert-guarded (kalau salah, parse_model gagal keras karena channel Concat tak cocok,
# bukan diam-diam salah). Sudah diverifikasi offline: save-layers hasil parse_model =
# [3,7,10,13,16,19,22], cocok persis perhitungan manual.
_lines = base_yaml.split("\n")
_t5 = None
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 5") and "VSSBlock" in _l and "Deform" not in _l:
        assert _t5 is None, "posisi 5: >1 baris cocok"
        _t5 = _i
assert _t5 is not None, "posisi 5 tidak ditemukan"
_dsd = [
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5a (D)",
    "  - [-1, 1, VSSBlock, [512]]             # 5b (S)",
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5c (D)",
]
_lines = _lines[:_t5] + _dsd + _lines[_t5+1:]
_hit7 = 0
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 7") and "VSSBlock" in _l and "Deform" not in _l:
        _lines[_i] = _l.replace("VSSBlock", "DeformVSSBlock")
        _hit7 += 1
assert _hit7 == 1, f"posisi 7: {_hit7}x"
_text = "\n".join(_lines)
_reindex = [
    ("  - [[-1, 5], 1, Concat, [1]]  # cat backbone P4",
     "  - [[-1, 7], 1, Concat, [1]]  # cat backbone P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 11], 1, Concat, [1]]  # cat head P4",
     "  - [[-1, 13], 1, Concat, [1]]  # cat head P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 8], 1, Concat, [1]]  # cat head P5",
     "  - [[-1, 10], 1, Concat, [1]]  # cat head P5 (geser +2 krn split posisi 5)"),
    ("  - [[14, 17, 20], 1, Detect, [nc]]  # Detect(P3, P4, P5)",
     "  - [[16, 19, 22], 1, Detect, [nc]]  # Detect(P3, P4, P5) (geser +2 krn split posisi 5)"),
]
for _old, _new in _reindex:
    assert _text.count(_old) == 1, f"GAGAL cari saat reindex: {_old!r}"
    _text = _text.replace(_old, _new)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-bb-late-alt.yaml", "w").write(_text)
print("  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki")

# Tulis ulang data.yaml SWRD v1 ke dalam repo dgn path ABSOLUT. data.yaml bawaan dataset
# pakai path relatif ("images/train", tanpa 'path:') yg hanya benar kalau cwd persis di folder
# dataset -- di sini cwd = REPO, jadi WAJIB dikonversi. nc & names diambil apa adanya dari
# file bawaan (TIDAK di-hardcode) supaya tidak salah kalau daftar kelas beda dari dugaan.
import yaml as _yaml
_d = _yaml.safe_load(open(SWRD_YAML_SRC))
assert "names" in _d, f"data.yaml SWRD tanpa 'names': {list(_d)}"
_names = _d["names"]
if isinstance(_names, dict):
    _names = [_names[k] for k in sorted(_names)]
_nc = _d.get("nc", len(_names))
assert _nc == len(_names), f"nc({_nc}) != jumlah names({len(_names)})"
assert _names == SWRD_NAMES_EXPECTED, f"names beda dari tripwire Sel 2: {_names}"

def _abs_split(v):
    """data.yaml bawaan bisa menyebut split sbg path relatif thd folder dataset."""
    if v is None:
        return None
    p = str(v)
    if os.path.isabs(p) and os.path.exists(p):
        return p
    for cand in (os.path.join(SWRD_BASE, p.lstrip("./")),
                 os.path.normpath(os.path.join(SWRD_BASE, p))):
        if os.path.exists(cand):
            return cand
    return None

_train = _abs_split(_d.get("train"))
_val   = _abs_split(_d.get("val")) or _abs_split(_d.get("valid"))
assert _train and _val, (
    f"Split train/val SWRD tidak ketemu. train={_d.get('train')} val={_d.get('val')} "
    f"-- cek listing folder di Sel 2 lalu perbaiki manual."
)
SWRD_NC = _nc
SWRD_YAML = f"{REPO}/ultralytics/cfg/datasets/swrdv1.yaml"
with open(SWRD_YAML, "w") as _f:
    _yaml.safe_dump({"train": _train, "val": _val, "nc": _nc, "names": _names},
                    _f, sort_keys=False, allow_unicode=True)
print(f"  OK swrdv1.yaml ditulis: nc={_nc}, names={_names}")
print(f"     train={_train}")
print(f"     val  ={_val}")

for f in ["ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py",
          "ultralytics/nn/tasks.py", "ultralytics/utils/torch_utils.py"]:
    py_compile.compile(f"{REPO}/{f}", doraise=True)
print("  OK sintaks semua file valid")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)
nproc = int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip() or "2")
os.environ["MAX_JOBS"] = str(max(1, nproc))

# Cache wheel selective_scan ke Drive. KUNCI CACHE WAJIB memuat versi CUDA torch:
# torch "2.11.0" bisa dibangun utk CUDA 12 (cu128) maupun CUDA 13 (cu130) dgn nomor versi SAMA.
# Kunci lama hanya "torch2.11.0" -> wheel hasil compile CUDA 12 (sesi lalu) dipasang ke torch CUDA 13
# -> ImportError libcudart.so.12. Kunci baru memisahkan keduanya; wheel lama tetap di Drive (tak dipakai).
import shutil as _sh
cap = torch.cuda.get_device_capability(0)
_tv = torch.__version__.replace("+", "-")
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{_tv}_cuda{torch.version.cuda}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"
os.makedirs(SS_WHEEL_DIR, exist_ok=True)

def _nvcc_version():
    nvcc = os.path.join(os.environ.get("CUDA_HOME", "/usr/local/cuda"), "bin", "nvcc")
    if not os.path.exists(nvcc):
        nvcc = _sh.which("nvcc") or nvcc
    try:
        out = subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout
        return re.search(r"release (\d+)\.(\d+)", out).group(0).split()[-1]
    except Exception:
        return None

def _ss_import_ok():
    # dicek di proses TERPISAH: import yang gagal tidak meninggalkan modul rusak di kernel ini
    r = subprocess.run([sys.executable, "-c", "import torch, selective_scan_cuda_core"], capture_output=True, text=True)
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]

def _ss_install(wheels):
    # --force-reinstall --no-deps: nama & versi wheel sama dgn yg (mungkin) sudah terpasang -> tanpa ini pip melewatinya
    subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps"] + wheels, check=True)

print(f"torch {torch.__version__} | CUDA torch {torch.version.cuda} | nvcc {_nvcc_version()} | {SS_TAG}")
cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
# Wheel LAMA (kunci tanpa versi CUDA, mis. 'sm80_torch2.11.0_py313') dibuat di sesi torch 2.11.0+cu128.
# Hanya dipakai bila torch sekarang juga build cu128, dan tetap WAJIB lolos uji import di bawah.
_legacy = f"{DRIVE_ROOT}/wheels/selective_scan_sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
if not cached_ss and torch.__version__.endswith("+cu128") and glob.glob(f"{_legacy}/*.whl"):
    for _w in glob.glob(f"{_legacy}/*.whl"):
        _sh.copy(_w, SS_WHEEL_DIR)
    cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    print(f"Wheel lama ditemukan di {os.path.basename(_legacy)} (cu128) -- disalin ke kunci baru, akan diuji import")
if cached_ss:
    print(f"Wheel selective_scan cache ditemukan ({SS_TAG}) -- install langsung, skip compile")
    _ss_install(cached_ss)
    ok, err = _ss_import_ok()
    if not ok:
        print(f"  Wheel cache GAGAL di-import ({err}) -- dihapus, compile ulang")
        for w in cached_ss: os.remove(w)
        cached_ss = []
if not cached_ss:
    _nv = _nvcc_version()
    assert _nv is not None, "nvcc tidak ditemukan -- tidak bisa compile selective_scan."
    assert _nv.split(".")[0] == str(torch.version.cuda).split(".")[0], (
        f"CUDA MAJOR berbeda: torch dibangun utk CUDA {torch.version.cuda}, nvcc sistem {_nv}. "
        f"Tidak ada wheel cache utk GPU ini ({SS_TAG}). Pakai GPU A100 (sama dgn run sebelumnya) atau kirim pesan ini ke Claude.")
    # CUDA 13 menghapus dukungan arsitektur < sm_75 (Maxwell/Pascal/Volta). setup.py selective_scan
    # (warisan VMamba) meng-compile sm_70 -> nvcc 13 menolak 'compute_70'. Ganti ke sm_75 HANYA bila CUDA>=13.
    # Tidak mengubah numerik di A100: kernel yg dipakai GPU sm_80 tetap cubin sm_80. (sm_75 = T4.)
    _setup = f"{REPO}/selective_scan/setup.py"
    if int(str(torch.version.cuda).split(".")[0]) >= 13:
        _ss = open(_setup).read()
        _n70 = len(re.findall(r"compute_70\s*,\s*code\s*=\s*sm_70", _ss))
        if _n70:
            _ss = re.sub(r"compute_70\s*,\s*code\s*=\s*sm_70", "compute_75,code=sm_75", _ss)
            open(_setup, "w").write(_ss)
        print(f"  setup.py selective_scan: {_n70} target sm_70 diganti sm_75 (CUDA {torch.version.cuda}); "
              f"arch tersisa: {sorted(set(re.findall(r'sm_(\d+)', _ss)))}")
    print(f"Belum ada cache utk {SS_TAG} -- compile sekali ini (~beberapa menit), lalu disimpan ke Drive")
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-v", "-w", SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    new_ss_wheels = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    _ss_install(new_ss_wheels)
    ok, err = _ss_import_ok()
    assert ok, f"selective_scan hasil compile tetap gagal di-import: {err}"
    print(f"Tersimpan ke {SS_WHEEL_DIR} -- sesi berikutnya dgn kombinasi GPU/torch/CUDA yg sama langsung cepat")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)

for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]:
    del sys.modules[k]
import selective_scan_cuda_core
print("selective_scan_cuda_core berhasil di-import.")
ENV_INFO = dict(torch=torch.__version__, cuda=torch.version.cuda, nvcc=_nvcc_version(), python=sys.version.split()[0],
                gpu=torch.cuda.get_device_name(0), ss_tag=SS_TAG)
print("Lingkungan run (catat di tracker):", ENV_INFO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn", "thop", "timm", "einops"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "wandb", "ray", "tensorboard", "tensorboardX", "tensorboard-data-server",
                "clearml", "comet_ml", "dvclive", "mlflow", "neptune"])

# (Blok penulisan dsf-yolo.yaml dari notebook DSF DIHAPUS -- di fase ini dataset-nya
# SWRD, dan yaml-nya sudah dibuat di atas sbg SWRD_YAML dgn nc/names yang dibaca
# dari data.yaml bawaan dataset, bukan di-hardcode.)

print("\nSEMUA PATCH + BUILD OK.")

Tripwire deform_ss2d.py OK

### Patch
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
  ~ 2.1: skip
  OK 3.1
  OK 3.2
  OK 3.3
  OK 4.1
  OK 4.2
  OK 5.1
  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate
  OK 3 YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate
  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki
  OK swrdv1.yaml ditulis: nc=6, names=['porosity', 'inclusion', 'crack', 'undercut', 'lack_of_fusion', 'lack_of_penetration']
     train=/content/data/swrd_v1/images/train
     val  =/content/data/swrd_v1/images/val
  OK sintaks semua file valid
torch 2.11.0+cu128 | CUDA torch 12.8 | nvcc 13.0 | sm80_torch2.11.0-cu128_cuda12.8_py313
Wheel selective_scan cache ditemukan (sm80_torch2.11.0-cu128_cuda12.8_py313) -- install langsung, skip compile
selective_scan_cuda_core berhasil di-import.
Lingkungan run (catat di tracker): {'torch': '2.11.0+cu128', 'cuda': '12.8', 'nvcc': '13.0', 'python': '3.13.15', 'gpu': 'NVIDIA A100-S

## 5. Pre-flight — Verifikasi Arsitektur

In [ ]:
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from ultralytics import settings
settings.update({"tensorboard": False})

from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules.deform_ss2d import DeformSS2D
from ultralytics.nn.modules.mamba_yolo import SS2D

cfg_path = f"ultralytics/cfg/models/mamba-yolo/{CONFIG_YAML}"
assert SWRD_NC == 6, f"SWRD v1 harus 6 kelas, dapat {SWRD_NC}"
m = DetectionModel(cfg_path, ch=3, nc=SWRD_NC, verbose=False)  # SWRD v1 = 6 kelas, BUKAN 8 spt DSF
n_par = sum(x.numel() for x in m.parameters())
n_def = sum(1 for x in m.modules() if isinstance(x, DeformSS2D))
n_plain = sum(1 for x in m.modules() if isinstance(x, SS2D) and not isinstance(x, DeformSS2D))

print(f"DATASET={DATASET} (nc={SWRD_NC})  MODEL={MODEL}  SEED={SEED}  config={CONFIG_YAML}")
# CATATAN: EXPECTED_PARAMS dihitung pada nc=8 (DSF-YOLO). Di SWRD v1 nc=6, jadi head Detect
# lebih kecil -> total params WAJAR sedikit di bawah target itu. Yang dicek ketat: jumlah
# DeformSS2D (penentu identitas arsitektur) + selisih params masih masuk akal (<0.05M).
_delta = n_par/1e6 - EXPECTED_PARAMS
print(f"Params: {n_par/1e6:.3f}M (acuan nc=8: {EXPECTED_PARAMS}M, selisih {_delta:+.3f}M -- "
      f"wajar negatif krn nc={SWRD_NC}<8)  DeformSS2D: {n_def} (target {EXPECTED_NDEF})")
assert n_def == EXPECTED_NDEF, f"Jumlah DeformSS2D {n_def} != {EXPECTED_NDEF}. STOP."
assert -0.05 < _delta < 0.01, f"Selisih params {_delta:+.3f}M di luar dugaan. STOP."

del m
torch.cuda.empty_cache()
print("Pre-flight LOLOS.")

Overriding model.yaml nc=80 with nc=6
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/c

DATASET=swrdv1 (nc=6)  MODEL=P1  SEED=5  config=DeMamba-YOLO-T.yaml
Params: 5.930M (acuan nc=8: 5.931M, selisih -0.001M -- wajar negatif krn nc=6<8)  DeformSS2D: 10 (target 10)
Pre-flight LOLOS.


## 6. Cek Status Checkpoint — Baru / Belum Selesai / Sudah Selesai

Sel ini yang membuat notebook aman dijalankan ulang. Tidak ada training di sini, cuma
pengecekan status.

In [ ]:
from pathlib import Path

last_ckpt = Path(DRIVE_OUTPUT) / RUN_NAME / "weights" / "last.pt"

if not last_ckpt.exists():
    STATUS = "BARU"
    print(f"STATUS: {STATUS} -- belum ada checkpoint, training akan mulai dari nol.")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep = raw.get("epoch", -1)
    has_opt = raw.get("optimizer") is not None
    if has_opt and ep >= 0 and ep < TARGET_EPOCHS - 1:
        STATUS = "BELUM_SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint ada, epoch tersimpan={ep}, optimizer ada.")
        print(f"  -> Akan RESUME (genuine, LR schedule kontinu) dari epoch {ep+1}.")
    else:
        STATUS = "SELESAI"
        print(f"STATUS: {STATUS} -- checkpoint sudah final_eval (optimizer={has_opt}, epoch={ep}).")
        print("  -> Training TIDAK diulang. Lanjut ke Sel 8 untuk baca hasil.")

print(f"\nlast.pt: {last_ckpt}  (ada: {last_ckpt.exists()})")

STATUS: BARU -- belum ada checkpoint, training akan mulai dari nol.

last.pt: /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002/weights/last.pt  (ada: False)


## 7. Training — Mulai Baru atau Resume, Sesuai Status Sel 6

**`patience=0` WAJIB** — mematikan EarlyStopping. Proyek ini butuh jumlah epoch yang
**terkontrol ketat** (150, sama persis tiap run) untuk perbandingan B2 vs P1 yang adil.
Default Ultralytics `patience=100` membuat run berhenti otomatis begitu 100 epoch berlalu
tanpa perbaikan `fitness` — kalau dibiarkan, run yang berbeda bisa berhenti di jumlah epoch
berbeda-beda (satu model plateau lebih cepat dari yang lain), merusak prinsip
compute-matched/epoch-matched yang jadi dasar seluruh perbandingan proyek ini.

In [ ]:
if STATUS == "BARU":
    model = YOLO(cfg_path)
    model.train(
        data=SWRD_YAML,   # <- dataset SWRD v1 (swrdv1.yaml, path absolut, dibuat di Sel 4)
        epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0,   # WAJIB: matikan EarlyStopping, perbandingan butuh epoch terkontrol ketat
        cache=True,   # percepat: dataset (3.400 tile PNG) muat di RAM, hindari re-decode tiap
                      # epoch. TIDAK mengubah data/hasil -- murni percepatan, aman utk perbandingan.
        optimizer="SGD",  # WAJIB: resep asli seed0 (mbyolo_train.py) pakai SGD, defaultnya.
                          # Tanpa ini, Ultralytics diam-diam pilih optimizer=auto (AdamW+LR
                          # auto) -- ditemukan sbg penyebab UTAMA divergensi P1 (bukan platform).
        cos_lr=USE_COS_LR,  # False (default eksperimen ini) -- kosinus TERBUKTI LEBIH BURUK,
                            # lihat catatan lengkap di Sel 3.
        lr0=LR0,  # WAJIB diturunkan dari default 0.01 -- lihat catatan lengkap di Sel 3.
                  # Ini yang menyerang akar masalah (nilai LR), bukan cuma bentuk kurvanya.
        # JANGAN PERNAH set time= -- itu override epochs= secara diam-diam (sama bahayanya dgn patience)
        project=DRIVE_OUTPUT, name=RUN_NAME,
    )
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    # TANPA epochs=/patience=/optimizer= -- resume mengambil train_args tersimpan di checkpoint
    # (150, patience=0, optimizer=SGD -- asalkan run aslinya sudah pakai settingan ini di atas).
    model.train(resume=True)
else:
    model = YOLO(str(last_ckpt))
    print("Training sudah selesai sebelumnya -- sel ini di-skip, tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if hasattr(model, "trainer") and model.trainer else Path(DRIVE_OUTPUT) / RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")

# Verifikasi status akhir -- kalau sesi ini SENDIRI terputus, baris ini tidak akan tercetak,
# dan menjalankan ulang notebook (dari Sel 1) akan mendeteksi STATUS=BELUM_SELESAI otomatis.
final_check = torch.load(str(Path(SAVE_DIR) / "weights" / "last.pt"), map_location="cpu", weights_only=False)
if final_check.get("optimizer") is None:
    print(f"Training {RUN_NAME} SELESAI PENUH ({TARGET_EPOCHS} epoch).")
else:
    print(f"Training belum selesai (epoch {final_check.get('epoch')}) -- jalankan ulang notebook ini untuk lanjut.")

WARNING ⚠️ no model scale passed. Assuming scale='T'.
New https://pypi.org/project/ultralytics/8.4.182 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml, data=/content/Mamba-YOLO/ultralytics/cfg/datasets/swrdv1.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=swrdv1_p1_seed5_150ep_lr0.002, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=5, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, h

100%|██████████| 755k/755k [00:00<00:00, 16.7MB/s]

Overriding model.yaml nc=80 with nc=6
WARNING ⚠️ no model scale passed. Assuming scale='T'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      5136  ultralytics.nn.modules.mamba_yolo.SimpleStem [3, 32, 3]                    
  1                  -1  1     29206  ultralytics.nn.modules.deform_ss2d.DeformVSSBlock[32, 32]                      
  2                  -1  1      8384  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[32, 64]                      
  3                  -1  1     96817  ultralytics.nn.modules.deform_ss2d.DeformVSSBlock[64, 64]                      
  4                  -1  1     33152  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[64, 128]                     
  5                  -1  3   1041231  ultralytics.nn.modules.deform_ss2d.DeformVSSBlock[128, 128]                    
  6                  -1  1    131840  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[1


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


DeMamba-YOLO-T summary: 544 layers, 5930479 parameters, 5930463 gradients, 14.1 GFLOPs

Freezing layer 'model.21.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


...

  0%|          | 0/338 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    150/150      5.85G      2.253      2.138       1.81          4        640: 100%|██████████| 338/338 [01:32<00:00,  3.67it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:05<00:00,  7.72it/s]

                   all        700        884      0.192      0.254      0.154     0.0543



150 epochs completed in 4.167 hours.
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002/weights/last.pt, 12.2MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002/weights/best.pt, 12.2MB

Validating /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002/weights/best.pt...
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
DeMamba-YOLO-T summary: 528 layers, 5929135 parameters, 0 gradients, 14.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:06<00:00,  7.31it/s]


                   all        700        884      0.182      0.197       0.16     0.0598
              porosity        209        619      0.454      0.583      0.517      0.215
             inclusion         54         78     0.0945     0.0513     0.0372     0.0113
                 crack         75         85       0.22      0.165      0.111     0.0255
              undercut         13         14      0.258      0.214      0.228     0.0929
        lack_of_fusion         37         47          0          0     0.0199    0.00514
   lack_of_penetration         40         41      0.065      0.171     0.0446    0.00907
Speed: 0.1ms preprocess, 5.6ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002

SAVE_DIR: /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002
Training swrdv1_p1_seed5_150ep_lr0.002 SELESAI PENUH (150 epoch).


## 8. Baca Hasil (Hanya Kalau Sudah Selesai Penuh)

In [ ]:
import pandas as pd

df = pd.read_csv(Path(SAVE_DIR) / "results.csv")
df.columns = df.columns.str.strip()
last = df.iloc[-1]

print(f"=== {RUN_NAME} — {len(df)} baris tercatat (epoch terakhir: {int(df['epoch'].max())}) ===")
print(f"  mAP50    = {last['metrics/mAP50(B)']:.4f}")
print(f"  mAP50:95 = {last['metrics/mAP50-95(B)']:.4f}")
print(f"  P={last['metrics/precision(B)']:.4f}  R={last['metrics/recall(B)']:.4f}")

# Cek epoch TERAKHIR yang tercapai, BUKAN jumlah baris -- results.csv kadang bolong di
# tengah (mis. Drive I/O sempat gagal nulis 1-2 baris) walau training-nya sendiri utuh
# tanpa gangguan. len(df) < TARGET_EPOCHS bisa salah nunjuk "belum selesai" padahal
# training sudah genap sampai epoch target -- ditemukan dari kasus B2 seed4 (146 baris
# tapi epoch max=150, gap di epoch 26-27 & 87-88, loss mulus tanpa lompatan di sekitarnya).
epoch_tercapai = int(df['epoch'].max())
n_gap = TARGET_EPOCHS - len(df)
if epoch_tercapai < TARGET_EPOCHS:
    print(f"\n⚠️  Epoch terakhir cuma {epoch_tercapai}/{TARGET_EPOCHS} -- training BELUM selesai.")
    print("   Jalankan ulang notebook ini (Runtime > Run all) untuk lanjut.")
else:
    print(f"\nSELESAI PENUH -- epoch terakhir tercapai {epoch_tercapai}/{TARGET_EPOCHS}.")
    if n_gap > 0:
        missing = sorted(set(range(1, TARGET_EPOCHS+1)) - set(df['epoch'].astype(int)))
        print(f"  Catatan: {n_gap} baris log bolong di tengah (epoch {missing}) -- kemungkinan")
        print(f"  Drive I/O sempat gagal nulis, BUKAN training terputus. Cek loss di sekitar")
        print(f"  epoch itu kalau ragu (harus mulus, tanpa lompatan/reset).")
    print(f"Checkpoint: {Path(SAVE_DIR) / 'weights' / 'best.pt'}")
    print("Catat path ini untuk arsip & analisis agregat multi-seed nanti.")


=== swrdv1_p1_seed5_150ep_lr0.002 — 150 baris tercatat (epoch terakhir: 150) ===
  mAP50    = 0.1545
  mAP50:95 = 0.0543
  P=0.1920  R=0.2537

SELESAI PENUH -- epoch terakhir tercapai 150/150.
Checkpoint: /content/drive/MyDrive/demamba-yolowd/output/swrdv1_p1_seed5_150ep_lr0.002/weights/best.pt
Catat path ini untuk arsip & analisis agregat multi-seed nanti.


## 9. Breakdown Per-Kelas

Angka agregat (Sel 8) tidak cukup untuk Ablasi F (apakah keuntungan P1 terkonsentrasi di kelas
elongated/diagonal seperti crack, walau kalah di rata-rata karena air-hole yang dominan jumlahnya).
Sel ini WAJIB dijalankan tiap run selesai -- jangan cuma andalkan angka agregat Sel 8.

Kolom `status_kelas`: **utama** (≥40 film val: porosity, inclusion, crack) atau **deskriptif** (LoP 38, LoF 29, undercut 9 film val) — aturan yg dikunci sebelum training.


In [ ]:
import pandas as pd

model_final = YOLO(str(Path(SAVE_DIR) / "weights" / "best.pt"))
mf = model_final.val(data=SWRD_YAML, split="val", verbose=False)

print(f"=== {RUN_NAME} -- per-kelas mAP50:95 ===")
print(f"Agregat: mAP50={mf.box.map50:.4f}  mAP50:95={mf.box.map:.4f}\n")
# AP50, P, R per kelas (urutannya mengikuti ap_class_index, bukan indeks kelas -- dipetakan eksplisit)
try:
    _idx = [int(i) for i in mf.box.ap_class_index]
    AP50 = dict(zip(_idx, [float(v) for v in mf.box.ap50]))
    PREC = dict(zip(_idx, [float(v) for v in mf.box.p]))
    REC  = dict(zip(_idx, [float(v) for v in mf.box.r]))
except Exception as _e:
    print(f"(AP50/P/R per kelas tidak tersedia di versi ultralytics ini: {_e})"); AP50, PREC, REC = {}, {}, {}
DESKRIPTIF = {"undercut", "lack_of_fusion", "lack_of_penetration"}   # <40 film val (dikunci sebelum training)
print(f"{'Kelas':20s} {'mAP50:95':>9s} {'AP50':>7s} {'P':>6s} {'R':>6s}  status")
for ci, cn in mf.names.items():
    print(f"{cn:20s} {mf.box.maps[ci]:>9.4f} {AP50.get(ci, float('nan')):>7.4f} {PREC.get(ci, float('nan')):>6.3f} {REC.get(ci, float('nan')):>6.3f}  "
          f"{'deskriptif' if cn in DESKRIPTIF else 'utama'}")

# Simpan ke CSV (append, bukan timpa) -- MODEL & SEED diambil OTOMATIS dari variabel
# Sel 3, bukan diketik ulang manual (itu sumber human-error sebelumnya: lupa update
# nomor seed saat pakai ulang snippet utk kombinasi lain).
# FILE TERPISAH dari DSF-YOLO. Kunci dedup CSV lama = model+seed+class; kalau SWRD ditulis
# ke file yg sama, baris "ALL" utk B2/P1 seed yg sama akan MENIMPA hasil DSF-YOLO diam-diam.
# Dua lapis pengaman: (1) nama file beda, (2) kolom "dataset" ikut jadi kunci dedup.
OUT_CSV = f"{DRIVE_ROOT}/analysis/per_class_results_swrdv1{'_diag' if DIAG_LR else ''}.csv"   # BUKAN _swrd.csv (= RF-Weld lama)
Path(f"{DRIVE_ROOT}/analysis").mkdir(parents=True, exist_ok=True)

rows = [{
    "dataset": CSV_TAG, "model": MODEL, "seed": SEED, "class": "ALL",
    "mAP50_95": mf.box.map, "mAP50": mf.box.map50,
    "precision": mf.box.mp, "recall": mf.box.mr,
    "status_kelas": "agregat", "deform_sha16": DEFORM_SHA256[:16],
    "ckpt_path": str(Path(SAVE_DIR) / "weights" / "best.pt"),
}]
for ci, cn in mf.names.items():
    rows.append({
        "dataset": CSV_TAG, "model": MODEL, "seed": SEED, "class": cn,
        "mAP50_95": mf.box.maps[ci], "mAP50": AP50.get(ci),
        "precision": PREC.get(ci), "recall": REC.get(ci),
        "status_kelas": "deskriptif" if cn in DESKRIPTIF else "utama", "deform_sha16": DEFORM_SHA256[:16],
        "ckpt_path": str(Path(SAVE_DIR) / "weights" / "best.pt"),
    })
df_new = pd.DataFrame(rows)

if Path(OUT_CSV).exists():
    df_old = pd.read_csv(OUT_CSV)
    df_out = pd.concat([df_old, df_new], ignore_index=True)
    df_out = df_out.drop_duplicates(subset=["dataset", "model", "seed", "class"], keep="last")
else:
    df_out = df_new
df_out.to_csv(OUT_CSV, index=False)
print(f"\nTersimpan ke: {OUT_CSV}  ({len(df_out)} baris total, {DATASET}/{MODEL} seed{SEED})")


Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
DeMamba-YOLO-T summary: 528 layers, 5929135 parameters, 0 gradients, 14.1 GFLOPs


val: Scanning /content/data/swrd_v1/labels/val.cache... 700 images, 350 backgrounds, 0 corrupt: 100%|██████████| 700/700 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:07<00:00,  5.70it/s]


                   all        700        884      0.184      0.197       0.16     0.0579
Speed: 0.1ms preprocess, 8.0ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/Mamba-YOLO/runs/detect/val
=== swrdv1_p1_seed5_150ep_lr0.002 -- per-kelas mAP50:95 ===
Agregat: mAP50=0.1597  mAP50:95=0.0579

Kelas                 mAP50:95    AP50      P      R  status
porosity                0.2148  0.5151  0.453  0.580  utama
inclusion               0.0116  0.0380  0.094  0.051  utama
crack                   0.0256  0.1113  0.230  0.165  utama
undercut                0.0811  0.2278  0.260  0.214  deskriptif
lack_of_fusion          0.0051  0.0201  0.000  0.000  deskriptif
lack_of_penetration     0.0092  0.0460  0.064  0.171  deskriptif

Tersimpan ke: /content/drive/MyDrive/demamba-yolowd/analysis/per_class_results_swrdv1.csv  (70 baris total, swrdv1/P1 seed5)


## 9b. Diagnostik — `best.pt` di split TRAIN vs VAL (bukan bagian protokol)

Membedakan dua penyebab performa rendah: **overfitting** (mAP train jauh di atas val → data kurang) vs
**underfitting / target sulit** (mAP train ≈ val → model belum bisa menyesuaikan diri bahkan di data latihnya).
Evaluasi tanpa augmentasi, setting sama dengan Sel 9. **Tidak** mengubah training dan **tidak** menulis ke
`per_class_results_swrdv1.csv`; hasil ke `analysis/diag_train_vs_val_swrdv1.csv`. ±2–3 menit di A100.

Untuk run yang sudah selesai: set MODEL/SEED di Sel 3 → Run all (Sel 7 otomatis skip training).


In [ ]:
import pandas as pd

_m = YOLO(str(Path(SAVE_DIR) / "weights" / "best.pt"))
_rows = []
for _sp in ("train", "val"):
    _r = _m.val(data=SWRD_YAML, split=_sp, verbose=False, plots=False)
    _idx = [int(i) for i in _r.box.ap_class_index]
    _ap50 = dict(zip(_idx, [float(v) for v in _r.box.ap50]))
    _rows.append({"dataset": CSV_TAG, "model": MODEL, "seed": SEED, "split": _sp, "class": "ALL",
                  "mAP50_95": float(_r.box.map), "mAP50": float(_r.box.map50)})
    for ci, cn in _r.names.items():
        _rows.append({"dataset": CSV_TAG, "model": MODEL, "seed": SEED, "split": _sp, "class": cn,
                      "mAP50_95": float(_r.box.maps[ci]), "mAP50": _ap50.get(ci, float("nan"))})
_d = pd.DataFrame(_rows)
_t = _d[_d.split == "train"].set_index("class"); _v = _d[_d.split == "val"].set_index("class")
print(f"=== {RUN_NAME} -- best.pt: TRAIN vs VAL (tanpa augmentasi) ===")
print(f"{'kelas':20s} {'train 50:95':>11s} {'val 50:95':>10s} {'train AP50':>10s} {'val AP50':>9s}")
for cn in _t.index:
    print(f"{cn:20s} {_t.loc[cn,'mAP50_95']:>11.4f} {_v.loc[cn,'mAP50_95']:>10.4f} {_t.loc[cn,'mAP50']:>10.4f} {_v.loc[cn,'mAP50']:>9.4f}")
_g = _t.loc["ALL", "mAP50_95"] / max(_v.loc["ALL", "mAP50_95"], 1e-9)
print(f"\nRasio mAP50:95 train/val = {_g:.2f}  "
      "(>~2 condong overfitting -> data kurang; ~1 condong underfitting / target sulit -> resep atau geometri label)")

DIAG_CSV = f"{DRIVE_ROOT}/analysis/diag_train_vs_val_swrdv1.csv"
if Path(DIAG_CSV).exists():
    _d = pd.concat([pd.read_csv(DIAG_CSV), _d], ignore_index=True).drop_duplicates(
        subset=["dataset", "model", "seed", "split", "class"], keep="last")
_d.to_csv(DIAG_CSV, index=False)
print(f"Tersimpan ke: {DIAG_CSV}")


Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
DeMamba-YOLO-T summary: 528 layers, 5929135 parameters, 0 gradients, 14.1 GFLOPs


val: Scanning /content/data/swrd_v1/labels/train.cache... 2700 images, 1350 backgrounds, 0 corrupt: 100%|██████████| 2700/2700 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 169/169 [00:23<00:00,  7.07it/s]


                   all       2700       3154       0.31      0.295      0.235     0.0973
Speed: 0.1ms preprocess, 6.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
DeMamba-YOLO-T summary: 528 layers, 5929135 parameters, 0 gradients, 14.1 GFLOPs


val: Scanning /content/data/swrd_v1/labels/val.cache... 700 images, 350 backgrounds, 0 corrupt: 100%|██████████| 700/700 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:06<00:00,  6.92it/s]

                   all        700        884      0.184      0.197       0.16     0.0579
Speed: 0.1ms preprocess, 6.7ms inference, 0.0ms loss, 0.8ms postprocess per image


=== swrdv1_p1_seed5_150ep_lr0.002 -- best.pt: TRAIN vs VAL (tanpa augmentasi) ===
kelas                train 50:95  val 50:95 train AP50  val AP50
ALL                       0.0973     0.0579     0.2346    0.1597
porosity                  0.2804     0.2148     0.6152    0.5151
inclusion                 0.0606     0.0116     0.1391    0.0380
crack                     0.0713     0.0256     0.1802    0.1113
undercut                  0.0761     0.0811     0.1807    0.2278
lack_of_fusion            0.0488     0.0051     0.1027    0.0201
lack_of_penetration       0.0466     0.0092     0.1897    0.0460

Rasio mAP50:95 train/val = 1.68  (>~2 condong overfitting -> data kurang; ~1 condong underfitting / target sulit -> resep atau geometri label)
Tersimpan ke: /content/drive/MyDrive/demamba-yolowd/analysis/diag_train_vs_val_swrdv1.csv


## Checklist

- [ ] Sel 0: torch = 2.11.0+cu128 (sesi baru: Run all → restart otomatis → Run all lagi)
- [ ] Sel 1-2: Drive mount, dataset `benyxu/swrd-dataset` terunduh, **tripwire v4.1 COCOK** (train & val), **tripwire `deform_ss2d.py` = GRID-FIX**, GPU terdeteksi
- [ ] Sel 3: konfigurasi MODEL/SEED benar, RUN_NAME diawali `swrdv1_`, **`DIAG_LR = False`** untuk 10 run utama
- [ ] Sel 4-5: patch + `selective_scan` ter-import + baris `Lingkungan run` dicatat + pre-flight LOLOS (nc=6, DeformSS2D 0 utk B2 / 10 utk P1)
- [ ] Sel 6: status checkpoint terbaca (BARU / BELUM_SELESAI / SELESAI) -- run pertama tiap kombinasi HARUS "BARU"
- [ ] Sel 7: training jalan sesuai status (baru/resume/skip)
- [ ] Sel 8: **150/150 epoch** — kalau belum, ulangi Sel 1-8 (Run all) sampai selesai
- [ ] Sel 9: breakdown per-kelas tercatat ke `per_class_results_swrdv1.csv` (WAJIB, jangan skip)
- [ ] Sel 9b (diagnostik): train vs val tercatat ke `diag_train_vs_val_swrdv1.csv`
- [ ] Centang baris di tabel progres (Sel 0)

**Catatan metodologi untuk paper:** semua 10 run SWRD v1 = 150 epoch kontinu, LR schedule tunggal, seed 1-5,
resep identik steel-pipe. Hasil run "swrd" lama (RF-Weld) TIDAK dicampur dengan "swrdv1".


## 10. Matikan Runtime (Wajib Dijalankan Terakhir)

In [ ]:
# Matikan runtime Colab otomatis setelah semua sel di atas selesai --
# supaya TIDAK terus memakan kuota Colab Pro kalau kamu tidak sempat
# matikan manual. Ini API resmi Google, benar-benar melepas VM-nya.
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)  # jeda singkat, jaga-jaga ada file yang masih proses ditulis ke Drive

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")
